# Pedestrian Evacuation Simulation

**Factorial computational study (research prototype)**
Author: Christian Somtoo Obiechina

This notebook runs a synthetic agent-based pedestrian evacuation experiment (24 conditions, 10 seeded replicates each, 240 simulations) and produces the results files in `./results`.

**Important:** All data are synthetic simulation outputs. The model is not calibrated or validated against real evacuation data, so the results describe this model under its stated assumptions only.

Dependencies: numpy, pandas, scipy, matplotlib. Run the cells in order, or use *Run All*.

## 1. Setup and design

In [1]:
import json
import math
import platform
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


In [2]:
# ----------------------------------------------------------------------------
# 1. Experimental design
# ----------------------------------------------------------------------------
CROWD_SIZES = [40, 80]            # agents
EXIT_WIDTHS = [0.8, 1.2, 1.6]     # metres
DESIRED_SPEEDS = [1.0, 1.3]       # metres per second (mean of the agent distribution)
OBSTACLE = [False, True]          # obstacle in front of the exit
REPLICATES = 10                   # random seeds per condition
BASE_SEED = 20260101

ROOM = 10.0          # square room, 10 m x 10 m
EXIT_X = 5.0         # exit centred on the top wall (y = 10)
DT = 0.05            # time step, seconds
T_MAX = 120.0        # simulation horizon, seconds
SEP_RADIUS = 0.8     # agents push apart inside this distance (m)
REPULSION = 1.5      # repulsion strength (m/s)
MAX_SPEED = 2.0      # speed cap (m/s)
OBS_CENTRE = (5.0, 8.2)
OBS_RADIUS = 0.5
OBS_INFLUENCE = 0.85


In [3]:
# ----------------------------------------------------------------------------
# 2. Agent model
# ----------------------------------------------------------------------------
def place_agents(n, rng, max_tries=20000):
    """Place n agents in the lower part of the room with a minimum separation of 0.4 m."""
    pts = []
    tries = 0
    while len(pts) < n:
        tries += 1
        if tries > max_tries:
            raise RuntimeError("Could not place agents; reduce crowd size or relax separation.")
        cand = np.array([rng.uniform(0.5, ROOM - 0.5), rng.uniform(0.5, 5.5)])
        if all(np.hypot(*(cand - p)) >= 0.4 for p in pts):
            pts.append(cand)
    return np.array(pts)


def simulate(n, width, desired_speed, obstacle, seed):
    """Run one evacuation. Returns per-agent evacuation times (NaN if not evacuated)."""
    rng = np.random.default_rng(seed)
    pos = place_agents(n, rng)
    v_des = np.clip(rng.normal(desired_speed, 0.1, n), 0.5, 1.8)
    active = np.ones(n, dtype=bool)
    evac_time = np.full(n, np.nan)
    t = 0.0

    while t < T_MAX and active.any():
        idx = np.where(active)[0]
        p = pos[idx]

        # Goal: the nearest point on the exit line, kept inside the exit opening.
        half = width / 2.0
        gx = np.clip(p[:, 0], EXIT_X - half + 0.15, EXIT_X + half - 0.15)
        goal = np.column_stack([gx, np.full(len(idx), ROOM)])
        delta = goal - p
        dist = np.linalg.norm(delta, axis=1)
        unit = delta / np.maximum(dist, 1e-9)[:, None]
        v = v_des[idx][:, None] * unit

        # Inter-agent repulsion
        if len(idx) > 1:
            diff = p[:, None, :] - p[None, :, :]
            r = np.linalg.norm(diff, axis=2)
            np.fill_diagonal(r, np.inf)
            close = r < SEP_RADIUS
            if close.any():
                mag = np.where(close, REPULSION * (SEP_RADIUS - r) / SEP_RADIUS, 0.0)
                safe_r = np.where(np.isfinite(r), r, 1.0)
                direction = diff / safe_r[:, :, None]
                v += (mag[:, :, None] * direction).sum(axis=1)

        # Obstacle avoidance: push agents radially away from a circular obstacle
        if obstacle:
            c = np.array(OBS_CENTRE)
            dc = p - c
            rc = np.linalg.norm(dc, axis=1)
            near = rc < OBS_RADIUS + OBS_INFLUENCE
            push = np.where(near, 2.0 * (OBS_RADIUS + OBS_INFLUENCE - rc) / OBS_INFLUENCE, 0.0)
            v += (push[:, None] * dc / np.maximum(rc, 1e-9)[:, None])

        # Speed cap
        s = np.linalg.norm(v, axis=1)
        v *= np.minimum(1.0, MAX_SPEED / np.maximum(s, 1e-9))[:, None]

        new_p = p + v * DT
        new_p[:, 0] = np.clip(new_p[:, 0], 0.2, ROOM - 0.2)
        new_p[:, 1] = np.clip(new_p[:, 1], 0.2, ROOM)
        pos[idx] = new_p
        t += DT

        at_exit = (new_p[:, 1] >= ROOM - 1e-9) & (np.abs(new_p[:, 0] - EXIT_X) <= half)
        done = idx[at_exit]
        evac_time[done] = t
        active[done] = False

    return evac_time


def summarise_run(evac_time, n):
    """Outcome measures for one run."""
    evacuated = np.isfinite(evac_time)
    n_evac = int(evacuated.sum())
    times = np.sort(evac_time[evacuated])
    target = math.ceil(0.9 * n)
    if n_evac >= target:
        t90 = float(times[target - 1])
        censored = False
    else:
        t90 = T_MAX          # right-censored at the horizon
        censored = True
    return {
        "t90_s": t90,
        "t90_censored": censored,
        "fraction_evacuated": n_evac / n,
        "mean_evacuation_time_s": float(times.mean()) if n_evac else np.nan,
        "throughput_per_s": n_evac / times[-1] if n_evac else 0.0,
        "n_evacuated": n_evac,
    }


In [4]:
# ----------------------------------------------------------------------------
# 3. Experiment
# ----------------------------------------------------------------------------
def run_experiment():
    rows = []
    agent_rows = []
    conditions = [
        (n, w, v, o)
        for n in CROWD_SIZES
        for w in EXIT_WIDTHS
        for v in DESIRED_SPEEDS
        for o in OBSTACLE
    ]
    for cond_id, (n, w, v, o) in enumerate(conditions):
        for rep in range(REPLICATES):
            # Same seed for the same replicate across conditions, so comparisons are paired.
            seed = BASE_SEED + rep
            evac_time = simulate(n, w, v, o, seed)
            m = summarise_run(evac_time, n)
            rows.append({
                "condition_id": cond_id,
                "n_agents": n,
                "exit_width_m": w,
                "desired_speed_mps": v,
                "obstacle": int(o),
                "replicate": rep,
                "seed": seed,
                **m,
            })
            for agent_id, t_e in enumerate(evac_time):
                agent_rows.append({
                    "condition_id": cond_id, "replicate": rep, "agent_id": agent_id,
                    "evacuated": bool(np.isfinite(t_e)),
                    "evacuation_time_s": None if not np.isfinite(t_e) else round(float(t_e), 3),
                })
    return pd.DataFrame(rows), pd.DataFrame(agent_rows)


In [5]:
# ----------------------------------------------------------------------------
# 4. Statistics
# ----------------------------------------------------------------------------
def holm(pvals):
    """Holm step-down adjustment."""
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj


def paired_test(df, factor, low, high, label):
    """Paired Wilcoxon signed-rank test of high minus low for one factor.

    Runs are matched on every other factor and on replicate seed, so each pair differs
    only in the factor under test.
    """
    all_factors = ["n_agents", "exit_width_m", "desired_speed_mps", "obstacle"]
    other = [f for f in all_factors if f != factor] + ["replicate"]
    a = df[df[factor] == low].set_index(other)["t90_s"].rename("low")
    b = df[df[factor] == high].set_index(other)["t90_s"].rename("high")
    joined = pd.concat([a, b], axis=1, join="inner").dropna()
    d = joined["high"] - joined["low"]
    stat, p = stats.wilcoxon(d, alternative="two-sided", zero_method="wilcox")
    return {
        "hypothesis": label,
        "comparison": f"{factor}: {high} minus {low}",
        "n_pairs": int(len(d)),
        "median_difference_s": float(np.median(d)),
        "mean_difference_s": float(d.mean()),
        "wilcoxon_p": float(p),
    }


def factorial_regression(df):
    """OLS on coded factors with main effects and two-way interactions for the outcome t90."""
    width_c = (df["exit_width_m"] - 1.2) / 0.4                 # -1, 0, 1
    crowd_c = np.where(df["n_agents"] == 80, 1.0, -1.0)
    speed_c = np.where(df["desired_speed_mps"] == 1.3, 1.0, -1.0)
    obst_c = np.where(df["obstacle"] == 1, 1.0, -1.0)
    names = ["intercept", "width", "crowd", "speed", "obstacle",
             "width:crowd", "width:speed", "width:obstacle",
             "crowd:speed", "crowd:obstacle", "speed:obstacle"]
    X = np.column_stack([
        np.ones(len(df)), width_c, crowd_c, speed_c, obst_c,
        width_c * crowd_c, width_c * speed_c, width_c * obst_c,
        crowd_c * speed_c, crowd_c * obst_c, speed_c * obst_c,
    ])
    y = df["t90_s"].values
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    sigma2 = (resid @ resid) / dof
    cov = sigma2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t_vals = beta / se
    p_vals = 2 * (1 - stats.t.cdf(np.abs(t_vals), dof))
    return pd.DataFrame({
        "term": names, "coefficient_s": beta, "std_error": se,
        "t": t_vals, "p_value": p_vals,
    })


In [6]:
# ----------------------------------------------------------------------------
# 5. Figures and outputs
# ----------------------------------------------------------------------------
def make_figure(df, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
    for ax, crowd in zip(axes, CROWD_SIZES):
        sub = df[df["n_agents"] == crowd]
        data, labels = [], []
        for w in EXIT_WIDTHS:
            data.append(sub[sub["exit_width_m"] == w]["t90_s"].values)
            labels.append(f"{w:.1f} m")
        ax.boxplot(data, tick_labels=labels)
        ax.set_title(f"{crowd} agents")
        ax.set_xlabel("Exit width")
        ax.grid(axis="y", alpha=0.3)
    axes[0].set_ylabel("Time to evacuate 90% (s)")
    fig.suptitle("Simulated time to 90% evacuation, pooled over speed and obstacle conditions\n"
                 "(synthetic model output, not validated against real evacuations)", fontsize=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def main():
    out = Path("results")
    out.mkdir(exist_ok=True)

    print("Running 24 conditions x 10 replicates = 240 simulations...")
    runs, agents = run_experiment()
    runs.to_csv(out / "evacuation_simulation_replicates.csv", index=False)
    agents.to_csv(out / "agent_level_results.csv", index=False)

    summary = (runs.groupby(["n_agents", "exit_width_m", "desired_speed_mps", "obstacle"])
               .agg(runs=("t90_s", "size"),
                    t90_mean_s=("t90_s", "mean"),
                    t90_sd_s=("t90_s", "std"),
                    fraction_evacuated_mean=("fraction_evacuated", "mean"),
                    throughput_mean_per_s=("throughput_per_s", "mean"),
                    censored_runs=("t90_censored", "sum"))
               .reset_index())
    summary.to_csv(out / "scenario_summary.csv", index=False)

    # Hypothesis tests (paired, Holm adjusted across the three main hypotheses)
    tests = [
        paired_test(runs, "exit_width_m", 0.8, 1.6, "H1: wider exits reduce t90"),
        paired_test(runs, "n_agents", 40, 80, "H2: larger crowds increase t90"),
        paired_test(runs, "obstacle", 0, 1, "H3: obstacle worsens t90"),
    ]
    p_adj = holm([t["wilcoxon_p"] for t in tests])
    for t, pa in zip(tests, p_adj):
        t["holm_p"] = float(pa)
    pd.DataFrame(tests).to_csv(out / "hypothesis_tests.csv", index=False)

    reg = factorial_regression(runs)
    reg.to_csv(out / "factorial_regression.csv", index=False)

    make_figure(runs, out / "t90_by_condition.png")

    meta = {
        "description": "Synthetic agent-based pedestrian evacuation simulation (research prototype)",
        "validated_against_real_data": False,
        "room_m": ROOM, "exit_centre_m": [EXIT_X, ROOM],
        "crowd_sizes": CROWD_SIZES, "exit_widths_m": EXIT_WIDTHS,
        "desired_speeds_mps": DESIRED_SPEEDS, "obstacle_conditions": OBSTACLE,
        "replicates": REPLICATES, "base_seed": BASE_SEED,
        "dt_s": DT, "t_max_s": T_MAX,
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": stats.__name__,
    }
    (out / "run_metadata.json").write_text(json.dumps(meta, indent=2))
    print("Done. Outputs in ./results")
    print(summary.to_string(index=False))
    print(pd.DataFrame(tests).to_string(index=False))
    print(reg.round(4).to_string(index=False))


## 2. Run the experiment

In [7]:
main()


Running 24 conditions x 10 replicates = 240 simulations...


Done. Outputs in ./results
 n_agents  exit_width_m  desired_speed_mps  obstacle  runs  t90_mean_s  t90_sd_s  fraction_evacuated_mean  throughput_mean_per_s  censored_runs
       40           0.8                1.0         0    10      10.615  0.293494                      1.0               3.370953              0
       40           0.8                1.0         1    10      11.660  0.532708                      1.0               2.857813              0
       40           0.8                1.3         0    10       7.940  0.188267                      1.0               4.529448              0
       40           0.8                1.3         1    10       8.530  0.240601                      1.0               4.005804              0
       40           1.2                1.0         0    10      10.260  0.265414                      1.0               3.482200              0
       40           1.2                1.0         1    10      11.265  0.403492                      1.0    